## Multi-task auxiliary objectives update
This notebook has been updated to work with the extended multi-task ASR head. You can enable or disable CTC, seq2seq, frame-level phoneme classifiers, speaker embeddings, and pronunciation error classifiers individually through `Configs/config.yml` under the `multi_task` section. Adjust the corresponding `loss_weights` entries when experimenting with these objectives.

# Auxiliary ASR Noise Robustness Evaluation

This notebook computes phoneme error rates (PER) for the auxiliary ASR model under clean conditions and when additive noise is applied to the input log-mel features.

In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            "type": "CUDA",
            "available": True,
            "name": device_name,
            "index": i
        })
else:
    devices.append({"type": "CUDA", "available": False, "name": "N/A"})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir("Configs"):
    %cd ../

!pwd

In [ ]:
# imports and helper utilities
import os
import yaml
import torch
import pandas as pd
from collections import Counter
from typing import Any, Callable, Dict, List, Optional, Sequence

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data

def cfg_get_nested(cfg: dict, path, default=None, sep: str = "."):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = list(path)

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def load_token_map_from_config(config: dict):
    token_src = config.get("phoneme_maps_path")
    if not token_src:
        return build_token_map_from_data(
            config.get("train_data"),
            config.get("val_data"),
            config.get("ood_data"),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}

def load_asr_model(model_path: str, config_path: str, device: torch.device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = cfg_get_nested(config, "model_params", {
        "input_dim": 80,
        "hidden_dim": 256,
        "n_token": len(token_map),
        "token_embedding_dim": 512,
        "n_layers": 5,
        "location_kernel_size": 31,
    })
    if "n_token" not in model_params:
        model_params["n_token"] = len(token_map)

    model_params.setdefault('stabilization_config', cfg_get_nested(config, 'stabilization', {}))

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("model", checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized)

    model.to(device)
    model.eval()
    return model, config, token_map

def build_dev_dataloader(config: dict, device: torch.device, batch_size: Optional[int] = None, num_workers: Optional[int] = None):
    val_csv_path = config.get("val_data")
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, "r", encoding="utf-8") as f:
        raw_lines = [line.rstrip("\n") for line in f]

    path_list: List[List[str]] = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split("|")
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ""
        else:
            text = "|".join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, "eval_params.batch_size", cfg_get_nested(config, "batch_size", 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, "dataloader_params.val_num_workers", 2))

    dataset_params = {
        "dict_path": cfg_get_nested(config, "phoneme_maps_path", "Data/word_index_dict.txt"),
        "sr": cfg_get_nested(config, "preprocess_params.sr", 24000),
        "spect_params": cfg_get_nested(
            config,
            "preprocess_params.spect_params",
            {"n_fft": 1024, "win_length": 1024, "hop_length": 300},
        ),
        "mel_params": cfg_get_nested(
            config,
            "preprocess_params.mel_params",
            {"n_mels": 80},
        ),
    }

    collate_config = {"return_wave": False}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader

NORMALIZATION_MEAN = -4.0
NORMALIZATION_STD = 4.0

In [ ]:
# locate checkpoint, load model, and prepare validation dataloader
checkpoint_dir = "Checkpoint"
if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith("epoch_") and f.endswith(".pth")]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split("_")[-1].split(".")[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

config_path = 'Checkpoint/config.yml'
if not os.path.isfile(config_path):
    raise FileNotFoundError(f"Config file '{config_path}' not found.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, config, token_map = load_asr_model(model_path, config_path, device)

phoneme_source = config.get("phoneme_maps_path", "built from dataset")
print(f"model --> {model_path}")
print(f"config --> {config_path}")
print(f"dictionary --> {phoneme_source}")

dev_loader = build_dev_dataloader(config, device)
print(f"Validation dataset size: {len(dev_loader.dataset)} samples")

vocab = token_map
if " " not in vocab:
    raise KeyError("The vocabulary does not contain the blank symbol ' '.")
BLANK_ID = vocab[" "]
ID2PH = {idx: symbol for symbol, idx in vocab.items()}
print(f"Blank token id: {BLANK_ID}")

In [ ]:
# decoding utilities, PER evaluation, and noise helpers
from typing import Dict


def ctc_greedy_decode(logits: torch.Tensor, lens: torch.Tensor) -> List[List[int]]:
    """Greedy CTC decoding with collapse and blank removal."""
    if logits.dim() != 3:
        raise ValueError(f"Expected logits of shape (B, T, V), got {tuple(logits.shape)}")

    pred_ids = logits.argmax(-1)
    hyps: List[List[int]] = []
    for b in range(pred_ids.size(0)):
        prev = BLANK_ID
        out: List[int] = []
        T = int(lens[b])
        for t in range(T):
            p = int(pred_ids[b, t])
            if p != BLANK_ID and p != prev:
                out.append(p)
            prev = p
        hyps.append(out)
    return hyps


def edit_distance(a: Sequence[int], b: Sequence[int]) -> int:
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + (a[i - 1] != b[j - 1]),
            )
    return dp[-1][-1]


def add_noise(mel: torch.Tensor, snr_db: float = 20.0) -> torch.Tensor:
    """Inject Gaussian noise into log-mel features by matching a target SNR."""
    mel_lin = mel.exp()
    sig_power = mel_lin.pow(2).flatten(1).mean(dim=1, keepdim=True)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise_std = noise_power.sqrt().view(-1, 1, 1)
    noise = torch.randn_like(mel_lin) * noise_std
    mel_noisy = (mel_lin + noise).clamp_min(1e-8).log()
    return mel_noisy


def make_noisy_transform(snr_db: float, mean: float = NORMALIZATION_MEAN, std: float = NORMALIZATION_STD) -> Callable[[torch.Tensor], torch.Tensor]:
    def transform(mels: torch.Tensor) -> torch.Tensor:
        mean_t = torch.tensor(mean, dtype=mels.dtype, device=mels.device)
        std_t = torch.tensor(std, dtype=mels.dtype, device=mels.device)
        mel_log = mels * std_t + mean_t
        mel_noisy = add_noise(mel_log, snr_db=snr_db)
        return (mel_noisy - mean_t) / std_t

    return transform


@torch.no_grad()
def run_per_evaluation(
    model: torch.nn.Module,
    dev_loader,
    device: Optional[torch.device] = None,
    mel_transform: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    max_examples: int = 5,
):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    tot_err, tot_len = 0, 0
    phoneme_freq: Counter = Counter()
    examples: List[Dict[str, Sequence[int]]] = []
    downsample_factor = 2 ** getattr(model, "n_down", 1)

    for batch in dev_loader:
        texts, text_lens, mels, mel_lens = batch[:4]
        texts = texts.to(torch.long)
        text_lens = text_lens.to(torch.long)
        mels = mels.to(device)
        mel_lens = mel_lens.to(torch.long)

        if mel_transform is not None:
            mels_in = mel_transform(mels)
        else:
            mels_in = mels

        outputs = model(mels_in)
        if isinstance(outputs, dict):
            logits = outputs.get("logits_ctc")
            if logits is None:
                raise KeyError("Model output dict does not contain 'logits_ctc'.")
        elif isinstance(outputs, (tuple, list)):
            logits = outputs[0]
        else:
            logits = outputs

        if logits.dim() != 3:
            raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

        logit_lens = torch.clamp(mel_lens // downsample_factor, min=1, max=logits.size(1))
        hyps = ctc_greedy_decode(logits.cpu(), logit_lens.cpu())

        for hyp, tgt, tgt_len in zip(hyps, texts, text_lens):
            effective_len = int(tgt_len)
            tgt_ids = tgt[:effective_len].tolist()
            tgt_ids = [idx for idx in tgt_ids if idx != BLANK_ID]

            phoneme_freq.update(tgt_ids)
            tot_err += edit_distance(hyp, tgt_ids)
            tot_len += len(tgt_ids)

            if len(examples) < max_examples:
                examples.append({
                    "prediction": hyp,
                    "reference": tgt_ids,
                })

    per = tot_err / max(1, tot_len)
    stats = {
        "total_errors": tot_err,
        "total_phonemes": tot_len,
        "phoneme_frequency": phoneme_freq,
        "examples": examples,
    }
    return per, stats


def eval_per(model: torch.nn.Module, dev_loader, device: Optional[torch.device] = None, max_examples: int = 5):
    return run_per_evaluation(model, dev_loader, device=device, mel_transform=None, max_examples=max_examples)


def per_with_noise(
    model: torch.nn.Module,
    dev_loader,
    snr_db: float = 20.0,
    device: Optional[torch.device] = None,
    max_examples: int = 5,
    mean: float = NORMALIZATION_MEAN,
    std: float = NORMALIZATION_STD,
):
    transform = make_noisy_transform(snr_db, mean=mean, std=std)
    return run_per_evaluation(model, dev_loader, device=device, mel_transform=transform, max_examples=max_examples)


def ids_to_symbols(ids: Sequence[int]) -> str:
    return " ".join(ID2PH.get(i, f"<unk:{i}>") for i in ids)


def display_examples(title: str, examples: List[Dict[str, Sequence[int]]], limit: int = 5):
    print(title)
    for idx, example in enumerate(examples[:limit], 1):
        print(f"Sample {idx}")
        print("Reference :", ids_to_symbols(example["reference"]))
        print("Prediction:", ids_to_symbols(example["prediction"]))
        print("-" * 60)


### Noise Validation
- small PER drift at 20 dB SNR is a good sign; massive jumps suggest the encoder is brittle.

In [ ]:
# Evaluate PER on clean validation features
clean_per, clean_stats = eval_per(model, dev_loader, device=device, max_examples=5)
print(f"PER (clean, CTC greedy): {clean_per:.3f}")
print(f"Total phonemes evaluated: {clean_stats['total_phonemes']}")
print(f"Total edit distance: {clean_stats['total_errors']}")

In [ ]:
display_examples("Clean validation examples", clean_stats["examples"])

In [ ]:
# Evaluate PER at several SNR levels
snr_levels = [20.0, 15.0, 10.0, 5.0, 0.0]
noise_results = []
for snr in snr_levels:
    per, _ = per_with_noise(model, dev_loader, snr_db=snr, device=device, max_examples=0)
    noise_results.append({"SNR (dB)": snr, "PER": per})
    print(f"PER @{snr:.1f} dB SNR: {per:.3f}")

noise_results_df = pd.DataFrame(noise_results)
noise_results_df

In [ ]:
# Inspect predictions at a selected SNR level
analysis_snr = 10.0
noisy_per, noisy_stats = per_with_noise(model, dev_loader, snr_db=analysis_snr, device=device, max_examples=5)
print(f"PER @{analysis_snr:.1f} dB SNR: {noisy_per:.3f}")
display_examples(f"Examples with additive noise @ {analysis_snr:.1f} dB SNR", noisy_stats["examples"])